# HMDA 2024 – Data Preparation (PySpark)

Reads the shared raw Parquet (`raw.parquet`), applies the same rules in `rules.py` as the Pandas version, creates S0/S1/S2 and writes them to `clean_spark/`.

- **Input:** `raw.parquet` (12,259,199 rows, 99 columns)
- **Outputs:** `clean_spark/` (S0, S1, S2, partitioned by `state_code`), `audit_spark/` (audit tables, cleaning log, comparison tables with Pandas, schema)
- Results match the Pandas version at every step (see the `*compare*.csv` files in `audit_spark/`).
- All cells have already been run; outputs are kept for reference.

## 0. Setup and Load Raw Parquet

In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/parquet
!cp -rn /content/drive/MyDrive/hmda/raw.parquet /content/parquet/

import os, sys, time
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("HMDA2024_PySpark")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "64")
         .getOrCreate())

BASE  = "/content/drive/MyDrive/hmda/"
AUDIT = BASE + "audit_spark/"
os.makedirs(AUDIT, exist_ok=True)

sys.path.insert(0, BASE)
import rules
print("Spark version:", spark.version)

Mounted at /content/drive
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
Spark version: 4.0.5


In [ ]:
KEEP = [
    "action_taken",
    "income", "debt_to_income_ratio", "loan_amount",
    "combined_loan_to_value_ratio", "property_value",
    "loan_type", "loan_purpose", "lien_status", "occupancy_type",
    "derived_loan_product_type", "construction_method",
    "applicant_age", "derived_race", "derived_ethnicity", "derived_sex",
    "denial_reason_1", "denial_reason_2", "denial_reason_3", "denial_reason_4",
    "state_code", "county_code", "census_tract", "lei",
    "tract_population", "tract_minority_population_percent",
    "ffiec_msa_md_median_family_income", "tract_to_msa_income_percentage",
    "tract_owner_occupied_units", "tract_one_to_four_family_homes",
    "tract_median_age_of_housing_units",
]

raw = spark.read.parquet("/content/parquet/raw.parquet")
print("Original columns:", len(raw.columns))
print("Columns not found:", [c for c in KEEP if c not in raw.columns])

t0 = time.time()
df = raw.select(*KEEP).cache()
total = df.count()                      # count() forces Spark to read and cache the data
load_time = time.time() - t0
print(f"Load + cache: {load_time:.1f}s | Rows: {total:,} | Columns: {len(df.columns)}")

Original columns: 99
Columns not found: []
Load + cache: 235.5s | Rows: 12,259,199 | Columns: 31


## 2. Data Understanding – Comparison with Pandas

In [ ]:
CODES = ["NA", "Exempt", "1111", "8888"]

exprs = []
for c in df.columns:
    col = F.col(c)
    exprs.append(F.sum(F.when(col.isNull() | (col == ""), 1).otherwise(0)).alias(f"{c}|null"))
    for k in CODES:
        exprs.append(F.sum(F.when(col == k, 1).otherwise(0)).alias(f"{c}|{k}"))

t0 = time.time()
r = df.agg(*exprs).first().asDict()
print(f"Time: {time.time()-t0:.1f}s")

miss = pd.DataFrame([{"column": c, **{k: r[f"{c}|{k}"] for k in ["null"] + CODES}}
                     for c in df.columns])
miss["pct_special"] = (miss[["null"] + CODES].sum(axis=1) / total * 100).round(2)
miss = miss.sort_values("pct_special", ascending=False).reset_index(drop=True)
miss.to_csv(AUDIT + "2c_missing_special_codes.csv", index=False)
miss

Time: 139.6s


,column,null,NA,Exempt,1111,8888,pct_special
0,denial_reason_4,12243362,0,0,0,0,99.87
1,denial_reason_3,12171698,0,0,0,0,99.29
2,denial_reason_2,11789224,0,0,0,0,96.17
3,debt_to_income_ratio,0,4060236,287240,0,0,35.46
4,combined_loan_to_value_ratio,0,4030609,287538,0,0,35.22
5,property_value,0,2455474,288601,0,0,22.38
6,income,0,1780434,0,77,11,14.52
7,applicant_age,0,0,0,0,1463684,11.94
8,census_tract,0,349023,0,0,0,2.85
9,county_code,0,296190,0,0,0,2.42


In [ ]:
t0 = time.time()
dup = total - df.dropDuplicates().count()
print(f"Duplicate rows: {dup:,} ({dup/total*100:.3f}%) | {time.time()-t0:.1f}s")

Duplicate rows: 71,299 (0.582%) | 237.9s


In [ ]:
act = df.groupBy("action_taken").count().orderBy(F.desc("count")).toPandas()
act["share_%"] = (act["count"] / total * 100).round(2)
act.to_csv(AUDIT + "2e_action_taken.csv", index=False)
display(act)

n1 = int(act.loc[act.action_taken == rules.ORIGINATED, "count"].sum())
n3 = int(act.loc[act.action_taken == rules.DENIED, "count"].sum())
print(f"Originated: {n1:,} | Denied: {n3:,}")
print(f"Baseline denial rate = {n3/(n1+n3)*100:.2f}%")

,action_taken,count,share_%
0,1,6197085,50.55
1,3,2103462,17.16
2,4,1538273,12.55
3,6,1276504,10.41
4,5,581940,4.75
5,2,361232,2.95
6,8,153391,1.25
7,7,47312,0.39


Originated: 6,197,085 | Denied: 2,103,462
Baseline denial rate = 25.34%


In [ ]:
NUM_COLS = ["income", "loan_amount", "combined_loan_to_value_ratio", "property_value"]
num = df.select(*[F.expr(f"try_cast({c} AS double)").alias(c) for c in NUM_COLS])

# Implausible values (proposal Section 5.1)
checks = num.agg(
    F.sum(F.when(F.col("income") <= 0, 1).otherwise(0)).alias("income_le_0"),
    F.sum(F.when(F.col("loan_amount") <= 0, 1).otherwise(0)).alias("loan_amount_le_0"),
    F.sum(F.when(F.col("combined_loan_to_value_ratio") > 250, 1).otherwise(0)).alias("cltv_gt_250"),
    F.sum(F.when(F.col("property_value") <= 0, 1).otherwise(0)).alias("property_value_le_0"),
).first().asDict()
implaus = pd.DataFrame([{"rule": k, "rows": v} for k, v in checks.items()])
implaus.to_csv(AUDIT + "2f_implausible_values.csv", index=False)
display(implaus)

# Statistics (percentile_approx – approximate percentiles)
stats = num.select(
    *[F.count(c).alias(f"{c}|n") for c in NUM_COLS],
    *[F.percentile_approx(c, [0.25, 0.5, 0.75], 100000).alias(f"{c}|q") for c in NUM_COLS],
).first().asDict()
num_summary = pd.DataFrame([{"column": c, "n": stats[f"{c}|n"],
                             "Q1": stats[f"{c}|q"][0], "median": stats[f"{c}|q"][1],
                             "Q3": stats[f"{c}|q"][2]} for c in NUM_COLS])
num_summary.to_csv(AUDIT + "2f_numeric_summary.csv", index=False)
num_summary

,rule,rows
0,income_le_0,165254
1,loan_amount_le_0,0
2,cltv_gt_250,3680
3,property_value_le_0,0


,column,n,Q1,median,Q3
0,income,10478765,67.000,104.00,165.00
1,loan_amount,12259199,105000.000,225000.00,375000.00
2,combined_loan_to_value_ratio,7941052,60.125,79.14,92.97
3,property_value,9515124,255000.000,385000.00,585000.00


In [ ]:
AP = BASE + "audit_pandas/"
AS = BASE + "audit_spark/"

# Pandas results (from audit_pandas files)
p_act = pd.read_csv(AP + "2e_action_taken.csv", dtype={"action_taken": str})
p_imp = pd.read_csv(AP + "2f_implausible_values.csv").set_index("rule")["rows"]
p_log = pd.read_csv(AP + "3_cleaning_log.csv")

# Spark results (from audit_spark files)
s_act = pd.read_csv(AS + "2e_action_taken.csv", dtype={"action_taken": str})
s_imp = pd.read_csv(AS + "2f_implausible_values.csv").set_index("rule")["rows"]
s_log = pd.read_csv(AS + "3_cleaning_log.csv")

def act_count(t, code):
    return int(t.loc[t["action_taken"] == code, "count"].sum())

def dup_count(log):
    return int(log.loc[log["step"] == "3.5 duplicates", "rows"].iloc[0])

expected = {
    "Total rows": (int(p_act["count"].sum()), int(s_act["count"].sum())),
    "Duplicate rows":   (dup_count(p_log), dup_count(s_log)),
    "Originated":   (act_count(p_act, "1"), act_count(s_act, "1")),
    "Denied":       (act_count(p_act, "3"), act_count(s_act, "3")),
    "income <= 0":  (int(p_imp["income_le_0"]), int(s_imp["income_le_0"])),
    "CLTV > 250":   (int(p_imp["cltv_gt_250"]), int(s_imp["cltv_gt_250"])),
}
cmp = pd.DataFrame([{"metric": k, "pandas (from file)": p, "spark (from file)": s, "match": p == s}
                    for k, (p, s) in expected.items()])
cmp.to_csv(AS + "2_compare_pandas_spark.csv", index=False)
cmp

,metric,pandas (from file),spark (from file),match
0,Total rows,12259199,12259199,True
1,Duplicate rows,71299,71299,True
2,Originated,6197085,6197085,True
3,Denied,2103462,2103462,True
4,income <= 0,165254,165254,True
5,CLTV > 250,3680,3680,True


## 3. Data Cleaning (applying rules.py)

In [ ]:
from pyspark.sql import Window

log = []
def add_log(step, column, rows, action):
    log.append({"step": step, "column": column, "rows": int(rows), "action": action})

t_clean = time.time()

In [ ]:
# Temporary row ID to define the "first" copy within each duplicate group
w = Window.partitionBy(*KEEP).orderBy("_rid")
base = (df.withColumn("_rid", F.monotonically_increasing_id())
          .withColumn("flag_duplicate", F.row_number().over(w) > 1)
          .drop("_rid"))

In [ ]:
def missing_cond(c):
    cond = F.col(c).isin(rules.MISSING_STRINGS)
    if c in rules.CODE_COLS:
        cond = cond | (F.col(c) == rules.EXEMPT_CODE)
    if c in rules.AGE_COLS:
        cond = cond | (F.col(c) == rules.AGE_NA_CODE)
    return cond

# Count affected rows per column (single pass)
cnt = df.agg(*[F.sum(F.when(missing_cond(c), 1).otherwise(0)).alias(c) for c in KEEP]).first().asDict()
for c in KEEP:
    if cnt[c]:
        add_log("3.2 missing codes", c, cnt[c], "NA/Exempt/1111/8888 -> NaN")

# Replace with null
clean = base.select(
    *[F.when(missing_cond(c), F.lit(None)).otherwise(F.col(c)).alias(c) for c in KEEP],
    "flag_duplicate")

pd.DataFrame(log)

,step,column,rows,action
0,3.2 missing codes,income,1780434,NA/Exempt/1111/8888 -> NaN
1,3.2 missing codes,debt_to_income_ratio,4347476,NA/Exempt/1111/8888 -> NaN
2,3.2 missing codes,combined_loan_to_value_ratio,4318147,NA/Exempt/1111/8888 -> NaN
3,3.2 missing codes,property_value,2744075,NA/Exempt/1111/8888 -> NaN
4,3.2 missing codes,applicant_age,1463684,NA/Exempt/1111/8888 -> NaN
5,3.2 missing codes,denial_reason_1,247789,NA/Exempt/1111/8888 -> NaN
6,3.2 missing codes,state_code,215946,NA/Exempt/1111/8888 -> NaN
7,3.2 missing codes,county_code,296190,NA/Exempt/1111/8888 -> NaN
8,3.2 missing codes,census_tract,349023,NA/Exempt/1111/8888 -> NaN


In [ ]:
# Count values that cannot be cast to numeric (Pandas: 0 in every column)
lost = clean.agg(*[
    F.sum(F.when(F.col(c).isNotNull() & F.expr(f"try_cast({c} AS double)").isNull(), 1)
           .otherwise(0)).alias(c)
    for c in rules.NUMERIC_COLS]).first().asDict()
for c, n in lost.items():
    if n:
        add_log("3.3 to numeric", c, n, "non-numeric value -> NaN")
print("Values lost when casting:", {c: n for c, n in lost.items() if n} or "none")

for c in rules.NUMERIC_COLS:
    clean = clean.withColumn(c, F.expr(f"try_cast({c} AS double)"))

Values lost when casting: none


In [ ]:
for c, (op, thr) in rules.IMPLAUSIBLE.items():
    cond = (F.col(c) <= thr) if op == "le" else (F.col(c) > thr)
    flag = f"flag_{c}_implausible"
    clean = (clean.withColumn(flag, F.coalesce(cond, F.lit(False)))
                  .withColumn(c, F.when(F.col(flag), F.lit(None)).otherwise(F.col(c))))

# Order columns exactly as in the Pandas output
FLAG_COLS = [f"flag_{c}_implausible" for c in rules.IMPLAUSIBLE] + ["flag_duplicate"]
clean = clean.select(*KEEP, *FLAG_COLS)

# Execute the full cleaning: cache, then count
clean = clean.cache()
n_clean = clean.count()
clean_time = time.time() - t_clean
df.unpersist()

# Count flags for the log
flags = clean.agg(*[F.sum(F.col(f).cast("int")).alias(f) for f in FLAG_COLS]).first().asDict()
for c, (op, thr) in rules.IMPLAUSIBLE.items():
    add_log("3.4 implausible", c, flags[f"flag_{c}_implausible"], f"{op} {thr} -> NaN (flagged)")
add_log("3.5 duplicates", "ALL", flags["flag_duplicate"], "flagged, not removed")

print(f"Cleaning time (Spark): {clean_time:.1f}s | Rows: {n_clean:,}")

Cleaning time (Spark): 784.3s | Rows: 12,259,199


In [ ]:
spark_log = pd.DataFrame(log)
spark_log.to_csv(AUDIT + "3_cleaning_log.csv", index=False)

pandas_log = pd.read_csv(BASE + "audit_pandas/3_cleaning_log.csv")
cmp3 = pandas_log.merge(spark_log, on=["step", "column"], how="outer",
                        suffixes=("_pandas", "_spark"))
cmp3["match"] = cmp3["rows_pandas"] == cmp3["rows_spark"]
cmp3 = cmp3[["step", "column", "rows_pandas", "rows_spark", "match"]]
cmp3.to_csv(AUDIT + "3_compare_cleaning_log.csv", index=False)
display(cmp3)

print(f"\nAll match: {cmp3['match'].all()}  ({cmp3['match'].sum()}/{len(cmp3)} rows)")
clean.printSchema()

,step,column,rows_pandas,rows_spark,match
0,3.2 missing codes,applicant_age,1463684,1463684,True
1,3.2 missing codes,census_tract,349023,349023,True
2,3.2 missing codes,combined_loan_to_value_ratio,4318147,4318147,True
3,3.2 missing codes,county_code,296190,296190,True
4,3.2 missing codes,debt_to_income_ratio,4347476,4347476,True
5,3.2 missing codes,denial_reason_1,247789,247789,True
6,3.2 missing codes,income,1780434,1780434,True
7,3.2 missing codes,property_value,2744075,2744075,True
8,3.2 missing codes,state_code,215946,215946,True
9,3.4 implausible,combined_loan_to_value_ratio,3680,3680,True



All match: True  (12/12 rows)
root
 |-- action_taken: string (nullable = true)
 |-- income: double (nullable = true)
 |-- debt_to_income_ratio: string (nullable = true)
 |-- loan_amount: double (nullable = true)
 |-- combined_loan_to_value_ratio: double (nullable = true)
 |-- property_value: double (nullable = true)
 |-- loan_type: string (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- lien_status: string (nullable = true)
 |-- occupancy_type: string (nullable = true)
 |-- derived_loan_product_type: string (nullable = true)
 |-- construction_method: string (nullable = true)
 |-- applicant_age: string (nullable = true)
 |-- derived_race: string (nullable = true)
 |-- derived_ethnicity: string (nullable = true)
 |-- derived_sex: string (nullable = true)
 |-- denial_reason_1: string (nullable = true)
 |-- denial_reason_2: string (nullable = true)
 |-- denial_reason_3: string (nullable = true)
 |-- denial_reason_4: string (nullable = true)
 |-- state_code: string (nulla

## 4. Feature Engineering

In [ ]:
from itertools import chain

t_fe = time.time()

# DTI lookup table from rules.py
dti_map = F.create_map(*[F.lit(x) for x in chain(*rules.DTI_BAND_MAP.items())])

# CLTV band: intervals (a, b] following rules.CLTV_UPPER / CLTV_BAND_ORDER
cltv = F.col("combined_loan_to_value_ratio")
cltv_band = None
for upper, label in zip(rules.CLTV_UPPER, rules.CLTV_BAND_ORDER):
    cltv_band = (F.when(cltv <= upper, label) if cltv_band is None
                 else cltv_band.when(cltv <= upper, label))

fe = (clean
    .withColumn("denied_flag", (F.col("action_taken") == rules.DENIED).cast("tinyint"))
    .withColumn("log_income", F.log("income"))
    .withColumn("log_loan_amount", F.log("loan_amount"))
    .withColumn("loan_to_income", F.col("loan_amount") / (F.col("income") * rules.INCOME_UNIT))
    .withColumn("dti_band", dti_map[F.col("debt_to_income_ratio")])
    .withColumn("cltv_band", cltv_band))

fe = fe.cache()
n_fe = fe.count()
fe_time = time.time() - t_fe
clean.unpersist()

print(f"Feature engineering time (Spark): {fe_time:.1f}s | Rows: {n_fe:,}")
fe.select("denied_flag", "log_income", "log_loan_amount",
          "loan_to_income", "dti_band", "cltv_band").printSchema()

Feature engineering time (Spark): 186.6s | Rows: 12,259,199
root
 |-- denied_flag: byte (nullable = true)
 |-- log_income: double (nullable = true)
 |-- log_loan_amount: double (nullable = true)
 |-- loan_to_income: double (nullable = true)
 |-- dti_band: string (nullable = true)
 |-- cltv_band: string (nullable = true)



In [ ]:
AP = BASE + "audit_pandas/"
cmp_bands = []

for c in ["dti_band", "cltv_band"]:
    s = fe.groupBy(c).count().toPandas()
    s[c] = s[c].fillna("NaN")

    p = pd.read_csv(AP + f"4_{c}.csv")
    p[c] = p[c].fillna("NaN").astype(str)

    m = p[[c, "count"]].merge(s, on=c, how="outer", suffixes=("_pandas", "_spark"))
    m["match"] = m["count_pandas"] == m["count_spark"]
    m.insert(0, "variable", c)
    m = m.rename(columns={c: "band"})
    cmp_bands.append(m)

cmp_bands = pd.concat(cmp_bands, ignore_index=True)
cmp_bands.to_csv(AUDIT + "4_compare_bands.csv", index=False)
display(cmp_bands)
print(f"All match: {cmp_bands['match'].all()}  ({cmp_bands['match'].sum()}/{len(cmp_bands)} rows)")

,variable,band,count_pandas,count_spark,match
0,dti_band,20-<30%,1070929,1070929,True
1,dti_band,30-<36%,1097751,1097751,True
2,dti_band,36-43%,2084592,2084592,True
3,dti_band,44-49%,1747928,1747928,True
4,dti_band,50-60%,825620,825620,True
5,dti_band,<20%,462987,462987,True
6,dti_band,>60%,621916,621916,True
7,dti_band,NaN,4347476,4347476,True
8,cltv_band,100-250,313124,313124,True
9,cltv_band,60-80,2946785,2946785,True


All match: True  (15/15 rows)


In [ ]:
FE_NUM = ["log_income", "log_loan_amount", "loan_to_income"]
QS = [0.01, 0.25, 0.5, 0.75, 0.99]

agg = fe.select(
    *[F.count(c).alias(f"{c}|count") for c in FE_NUM],
    *[F.mean(c).alias(f"{c}|mean") for c in FE_NUM],
    *[F.stddev(c).alias(f"{c}|std") for c in FE_NUM],
    *[F.min(c).alias(f"{c}|min") for c in FE_NUM],
    *[F.max(c).alias(f"{c}|max") for c in FE_NUM],
    *[F.percentile_approx(c, QS, 100000).alias(f"{c}|q") for c in FE_NUM],
).first().asDict()

p_sum = pd.read_csv(AP + "4_fe_numeric_summary.csv", index_col=0)

rows = []
for c in FE_NUM:
    sp = {"count": agg[f"{c}|count"], "mean": agg[f"{c}|mean"], "std": agg[f"{c}|std"],
          "min": agg[f"{c}|min"], "max": agg[f"{c}|max"],
          **{f"{int(q*100)}%": v for q, v in zip(QS, agg[f"{c}|q"])}}
    for stat, v in sp.items():
        pv = p_sum.loc[c, stat]
        exact = stat in ["count", "mean", "std", "min", "max"]
        tol = 0.001 if exact else 0.01          # the Pandas file is rounded to 3 decimals
        rows.append({"variable": c, "metric": stat, "pandas": pv, "spark": round(v, 3),
                     "difference": round(abs(pv - v), 4),
                     "type": "exact" if exact else "approximate",
                     "match": abs(pv - v) <= tol})

cmp_num = pd.DataFrame(rows)
cmp_num.to_csv(AUDIT + "4_compare_numeric.csv", index=False)
display(cmp_num)
print(f"All match: {cmp_num['match'].all()}  ({cmp_num['match'].sum()}/{len(cmp_num)} rows)")

,variable,metric,pandas,spark,difference,type,match
0,log_income,count,1.031351e+07,1.031351e+07,0.0000,exact,True
1,log_income,mean,4.689000e+00,4.689000e+00,0.0004,exact,True
2,log_income,std,7.430000e-01,7.430000e-01,0.0004,exact,True
3,log_income,min,0.000000e+00,0.000000e+00,0.0000,exact,True
4,log_income,max,2.072300e+01,2.072300e+01,0.0003,exact,True
5,log_income,1%,2.996000e+00,2.996000e+00,0.0003,approximate,True
6,log_income,25%,4.234000e+00,4.234000e+00,0.0001,approximate,True
7,log_income,50%,4.654000e+00,4.654000e+00,0.0000,approximate,True
8,log_income,75%,5.112000e+00,5.112000e+00,0.0000,approximate,True
9,log_income,99%,6.737000e+00,6.737000e+00,0.0000,approximate,True


All match: True  (30/30 rows)


In [ ]:
s1 = fe.filter(F.col("action_taken").isin(rules.S1_CODES))

for c, order in [("dti_band", rules.DTI_BAND_ORDER), ("cltv_band", rules.CLTV_BAND_ORDER)]:
    t = (s1.groupBy(c).agg(F.count("*").alias("n"),
                           F.round(F.mean("denied_flag") * 100, 2).alias("denial_rate"))
           .toPandas())
    t[c] = pd.Categorical(t[c], categories=order, ordered=True)
    t = t.dropna(subset=[c]).sort_values(c).reset_index(drop=True)
    t.to_csv(AUDIT + f"4_denial_rate_{c}.csv", index=False)
    print(f"\n=== Denial rate theo {c} (S1, Spark) ===")
    display(t)



=== Denial rate theo dti_band (S1, Spark) ===


,dti_band,n,denial_rate
0,<20%,432229,30.46
1,20-<30%,999770,16.65
2,30-<36%,1026750,14.83
3,36-43%,1951343,14.30
4,44-49%,1625066,15.67
5,50-60%,777872,40.04
6,>60%,603783,92.13



=== Denial rate theo cltv_band (S1, Spark) ===


,cltv_band,n,denial_rate
0,<=60,1879464,29.02
1,60-80,2783553,21.73
2,80-90,861463,25.41
3,90-95,556162,18.16
4,95-100,1081004,15.53
5,100-250,295411,30.19


## 5. Data Preparation – S0/S1/S2, write clean_spark, export schema

In [ ]:
S0 = fe
S1 = fe.filter(F.col("action_taken").isin(rules.S1_CODES))
S2 = fe.filter(F.col("action_taken").isin(rules.S2_CODES))

n0, n1s, n2s = S0.count(), S1.count(), S2.count()
s1_rate = S1.agg(F.mean("denied_flag")).first()[0] * 100

p_sizes = pd.read_csv(AP + "5_sample_sizes.csv")["rows"].tolist()
cmp5 = pd.DataFrame({
    "dataset": ["S0 (all)", "S1 (originated + denied)", "S2 (denied only)"],
    "pandas (from file)": p_sizes,
    "spark": [n0, n1s, n2s],
})
cmp5["match"] = cmp5["pandas (from file)"] == cmp5["spark"]
display(cmp5)
print(f"Denial rate trong S1 (Spark): {s1_rate:.2f}%")

,dataset,pandas (from file),spark,match
0,S0 (all),12259199,12259199,True
1,S1 (originated + denied),8300547,8300547,True
2,S2 (denied only),2103462,2103462,True


Denial rate trong S1 (Spark): 25.34%


In [ ]:
key = ["income", "debt_to_income_ratio", "combined_loan_to_value_ratio", "property_value"]

def miss_pct(d, n):
    r = d.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in key]).first().asDict()
    return {c: round(r[c] / n * 100, 2) for c in key}

m0, m1 = miss_pct(S0, n0), miss_pct(S1, n1s)
p_miss = pd.read_csv(AP + "5_missing_S0_vs_S1.csv", index_col=0)

cmp_miss = pd.DataFrame({
    "S0 pandas": p_miss["S0_missing_%"], "S0 spark": pd.Series(m0),
    "S1 pandas": p_miss["S1_missing_%"], "S1 spark": pd.Series(m1),
})
cmp_miss["match"] = ((cmp_miss["S0 pandas"] == cmp_miss["S0 spark"]) &
                    (cmp_miss["S1 pandas"] == cmp_miss["S1 spark"]))
cmp_miss.to_csv(AUDIT + "5_compare_missing.csv")
cmp_miss

,S0 pandas,S0 spark,S1 pandas,S1 spark,match
income,15.87,15.87,7.52,7.52,True
debt_to_income_ratio,35.46,35.46,10.65,10.65,True
combined_loan_to_value_ratio,35.25,35.25,10.16,10.16,True
property_value,22.38,22.38,6.44,6.44,True


In [ ]:
OUT = "/content/output_spark/"
!rm -rf {OUT}

write_times = {}
for name, d, n in [("S0_all", S0, n0), ("S1_orig_denied", S1, n1s), ("S2_denied", S2, n2s)]:
    t0 = time.time()
    (d.repartition("state_code")           # one file per state, as in the Pandas output
      .write.mode("overwrite")
      .partitionBy("state_code")
      .parquet(OUT + name))
    write_times[name] = round(time.time() - t0, 1)

    n_back = spark.read.parquet(OUT + name).count()
    print(f"{name}: write {write_times[name]}s | read back {n_back:,} rows | match: {n_back == n}")

!du -sh {OUT}*

S0_all: write 258.0s | read back 12,259,199 rows | match: True
S1_orig_denied: write 191.6s | read back 8,300,547 rows | match: True
S2_denied: write 51.0s | read back 2,103,462 rows | match: True
293M	/content/output_spark/S0_all
215M	/content/output_spark/S1_orig_denied
60M	/content/output_spark/S2_denied


In [ ]:
sch_spark  = {f.name: f.dataType.simpleString() for f in spark.read.parquet(OUT + "S0_all").schema}
sch_pandas = {f.name: f.dataType.simpleString()
              for f in spark.read.parquet(BASE + "clean_pandas/S0_all").schema}

cols = list(dict.fromkeys(list(sch_pandas) + list(sch_spark)))
schema = pd.DataFrame({
    "column": cols,
    "type_pandas": [sch_pandas.get(c, "—") for c in cols],
    "type_spark":  [sch_spark.get(c, "—") for c in cols],
})
schema["match"] = schema["type_pandas"] == schema["type_spark"]
schema.to_csv(AUDIT + "5_schema_compare.csv", index=False)

# Standalone schema file for the Spark output (deliverable)
schema[schema["type_spark"] != "—"][["column", "type_spark"]].to_csv(
    AUDIT + "schema_clean_spark.csv", index=False)

display(schema)
print("Columns present in only one engine:", schema.loc[~schema["match"], "column"].tolist())

,column,type_pandas,type_spark,match
0,action_taken,string,string,True
1,income,double,double,True
2,debt_to_income_ratio,string,string,True
3,loan_amount,double,double,True
4,combined_loan_to_value_ratio,double,double,True
5,property_value,double,double,True
6,loan_type,string,string,True
7,loan_purpose,string,string,True
8,lien_status,string,string,True
9,occupancy_type,string,string,True


Columns present in only one engine: ['sample_u']


In [ ]:
!mkdir -p {BASE}clean_spark
!cp -r {OUT}* {BASE}clean_spark/
!du -sh {BASE}clean_spark/*

for name, n in [("S0_all", n0), ("S1_orig_denied", n1s), ("S2_denied", n2s)]:
    got = spark.read.parquet(f"{BASE}clean_spark/{name}").count()
    print(f"{name}: {got:,} rows on Drive | correct: {got == n}")

293M	/content/drive/MyDrive/hmda/clean_spark/S0_all
215M	/content/drive/MyDrive/hmda/clean_spark/S1_orig_denied
60M	/content/drive/MyDrive/hmda/clean_spark/S2_denied
S0_all: 12,259,199 rows on Drive | correct: True
S1_orig_denied: 8,300,547 rows on Drive | correct: True
S2_denied: 2,103,462 rows on Drive | correct: True


In [ ]:
timing = pd.DataFrame([
    {"step": "Load + cache raw Parquet (31 columns)", "seconds": round(load_time, 1)},
    {"step": "Cleaning",                          "seconds": round(clean_time, 1)},
    {"step": "Feature engineering",               "seconds": round(fe_time, 1)},
] + [{"step": f"Write {k}", "seconds": v} for k, v in write_times.items()])
timing.to_csv(AUDIT + "5_spark_timing.csv", index=False)
timing

,step,seconds
0,Load + cache raw Parquet (31 columns),235.5
1,Cleaning,784.3
2,Feature engineering,186.6
3,Write S0_all,258.0
4,Write S1_orig_denied,191.6
5,Write S2_denied,51.0


## 6. Handover Check

In [ ]:
import pyarrow.dataset as ds

rows = []
for name in ["S0_all", "S1_orig_denied", "S2_denied"]:
    n = {eng: ds.dataset(f"{BASE}{eng}/{name}", format="parquet", partitioning="hive").count_rows()
         for eng in ["clean_pandas", "clean_spark"]}
    rows.append({"dataset": name, "pandas": n["clean_pandas"], "spark": n["clean_spark"],
                 "match": n["clean_pandas"] == n["clean_spark"]})

cmp5 = pd.DataFrame(rows)
cmp5.to_csv(BASE + "audit_spark/5_compare_sample_sizes.csv", index=False)
cmp5

,dataset,pandas,spark,match
0,S0_all,12259199,12259199,True
1,S1_orig_denied,8300547,8300547,True
2,S2_denied,2103462,2103462,True
